[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/33_computer_vision/03_detection_and_segmentation_concepts.ipynb)

# Bài 03 — Phát hiện vật thể và phân đoạn ảnh

**🎯 Mục tiêu**
- Phân biệt các bài toán: phân loại, định vị, **phát hiện** (detection), phân đoạn **ngữ nghĩa** / **thực thể** (semantic / instance segmentation).
- Bounding box và **IoU**; **NMS**; đánh giá bằng **precision–recall, AP, mAP** — cài from scratch.
- Tự xây một **detector dạng lưới** (ý tưởng YOLO) và train trên ảnh nhiều vật thể.
- Phân đoạn ngữ nghĩa với **U-Net**: vai trò của skip connection; vì sao pixel accuracy đánh lừa và cần **mIoU**.
- Bản đồ các họ mô hình thực tế (Faster R-CNN, YOLO, DETR, Mask R-CNN, SAM).

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/33_computer_vision

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Rectangle

from ml_course.datasets import SHAPE_CLASSES, make_shape_scenes
from ml_course.visualization import set_style

try:
    import torch
    import torch.nn.functional as F
    from torch import nn
except ImportError as e:
    raise ImportError("Chương này cần PyTorch: pip install -e \".[dl]\"") from e

set_style()
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cudnn.deterministic = True
NC = len(SHAPE_CLASSES)
print("thiết bị:", device)

## 1. Các bài toán thị giác

| Bài toán | Đầu ra | Ví dụ |
|---|---|---|
| Phân loại | 1 nhãn cho cả ảnh | ảnh này là "mèo" |
| Định vị | nhãn + 1 hộp | mèo ở hộp (x1, y1, x2, y2) |
| Phát hiện | **số lượng tuỳ ý** hộp + nhãn + điểm tin cậy | 3 người, 2 xe, mỗi cái một hộp |
| Phân đoạn ngữ nghĩa | nhãn cho **mỗi pixel** | pixel nào là đường, xe, người |
| Phân đoạn thực thể | mask riêng cho **từng** vật thể | người #1, người #2 |

Dữ liệu của bài: ảnh 64×64 chứa 1–3 hình không chồng nhau, kèm hộp bao và mask từng pixel.

In [ ]:
X_train, T_train, M_train = make_shape_scenes(4000, seed=0)
X_test, T_test, M_test = make_shape_scenes(500, seed=1)
X_train_t, X_test_t = torch.tensor(X_train[:, None]), torch.tensor(X_test[:, None])
print(f"train {X_train.shape}, test {X_test.shape} | số vật thể / ảnh (train): {np.bincount([len(t['labels']) for t in T_train])[1:]}")


def draw_boxes(ax, boxes, labels, color, scores=None):
    for k, ((x1, y1, x2, y2), c) in enumerate(zip(boxes, labels)):
        ax.add_patch(Rectangle((x1 - 0.5, y1 - 0.5), x2 - x1, y2 - y1, fill=False, color=color, lw=1.5))
        text = SHAPE_CLASSES[int(c)] + ("" if scores is None else f" {scores[k]:.2f}")
        ax.text(x1, y1 - 1.5, text, color=color, fontsize=7)


fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for j in range(4):
    axes[0, j].imshow(X_train[j], cmap="gray")
    draw_boxes(axes[0, j], T_train[j]["boxes"], T_train[j]["labels"], "red")
    axes[1, j].imshow(M_train[j], cmap="tab10", vmin=0, vmax=9, interpolation="nearest")
    axes[0, j].set_title("ảnh + hộp bao")
    axes[1, j].set_title("mask (0 = nền)")
for ax in axes.ravel():
    ax.axis("off")
plt.tight_layout()
plt.show()

## 2. Bounding box và IoU

Hộp thường biểu diễn dạng **xyxy** (góc trên-trái, góc dưới-phải), **xywh** hoặc **cxcywh** (tâm + kích thước) — đổi nhầm định dạng là lỗi phổ biến nhất khi dùng thư viện. Độ khớp giữa hai hộp đo bằng **Intersection over Union**:

$$\text{IoU}(A, B) = \frac{|A\cap B|}{|A\cup B|} \in [0, 1]$$

In [ ]:
def box_iou(a, b):
    """a: (N, 4), b: (M, 4) dạng xyxy → ma trận IoU (N, M)."""
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    x1 = np.maximum(a[:, None, 0], b[None, :, 0])
    y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2])
    y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    area = lambda r: (r[:, 2] - r[:, 0]) * (r[:, 3] - r[:, 1])
    return inter / (area(a)[:, None] + area(b)[None, :] - inter)


boxes = np.array([[0, 0, 10, 10], [5, 5, 15, 15], [0, 0, 10, 5], [20, 20, 30, 30]])
print(np.round(box_iou(boxes[:1], boxes), 3), "← cùng hộp, chồng 1/4, nửa hộp, không chồng")
try:
    from torchvision.ops import box_iou as tv_box_iou
    print(f"sai lệch so với torchvision.ops.box_iou: {np.abs(box_iou(boxes, boxes) - tv_box_iou(torch.tensor(boxes, dtype=torch.float), torch.tensor(boxes, dtype=torch.float)).numpy()).max():.1e}")
except ImportError:
    print("(không có torchvision — bỏ qua so sánh)")

Hai hộp lệch nhau chỉ 5 pixel theo mỗi chiều đã có IoU = 0.14: IoU rất khắt khe với vật nhỏ. Ngưỡng thông dụng để coi một dự đoán là "đúng": IoU ≥ 0.5 (PASCAL VOC); COCO lấy trung bình trên các ngưỡng 0.5, 0.55, ..., 0.95.

## 3. Một detector dạng lưới (ý tưởng YOLO)

Chia ảnh thành lưới 8×8 ô (mỗi ô 8 pixel). Ô chứa **tâm** của một vật thể chịu trách nhiệm dự đoán nó. Mỗi ô xuất $1 + 5 + 4$ số:
- **objectness**: có vật thể có tâm trong ô này không (sigmoid);
- **lớp**: 5 logits (softmax);
- **hộp**: vị trí tâm trong ô $(o_x, o_y) \in [0, 1]$ và kích thước $(w, h)$ theo tỉ lệ ảnh (sigmoid).

Một CNN thuần tích chập (3 lần pooling: 64 → 8) cho ra đúng tensor $8\times 8\times 10$. Loss = BCE (objectness, mọi ô) + cross-entropy (lớp, ô có vật) + L1 (hộp, ô có vật).

In [ ]:
IMG, GRID = 64, 8
CELL = IMG / GRID


def encode_targets(targets):
    obj = torch.zeros(len(targets), GRID, GRID)
    cls = torch.zeros(len(targets), GRID, GRID, dtype=torch.long)
    box = torch.zeros(len(targets), GRID, GRID, 4)
    for i, t in enumerate(targets):
        for (x1, y1, x2, y2), c in zip(t["boxes"], t["labels"]):
            cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
            gx, gy = int(cx // CELL), int(cy // CELL)
            obj[i, gy, gx] = 1
            cls[i, gy, gx] = int(c)
            box[i, gy, gx] = torch.tensor([cx / CELL - gx, cy / CELL - gy, (x2 - x1) / IMG, (y2 - y1) / IMG])
    return obj, cls, box


class GridDetector(nn.Module):
    def __init__(self):
        super().__init__()

        def block(i, o):
            return nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU())

        self.body = nn.Sequential(block(1, 32), nn.MaxPool2d(2), block(32, 64), nn.MaxPool2d(2),
                                  block(64, 96), nn.MaxPool2d(2), block(96, 128), block(128, 128))
        self.head = nn.Conv2d(128, 1 + NC + 4, kernel_size=1)

    def forward(self, x):
        return self.head(self.body(x)).permute(0, 2, 3, 1)             # (B, 8, 8, 10)


def detection_loss(out, obj, cls, box):
    pos = obj > 0
    loss_obj = F.binary_cross_entropy_with_logits(out[..., 0], obj)
    loss_cls = F.cross_entropy(out[..., 1:1 + NC][pos], cls[pos])
    loss_box = F.l1_loss(torch.sigmoid(out[..., 1 + NC:][pos]), box[pos])
    return loss_obj + loss_cls + 5 * loss_box


obj_t, cls_t, box_t = encode_targets(T_train)
torch.manual_seed(0)
detector = GridDetector().to(device)
opt = torch.optim.Adam(detector.parameters(), lr=1e-3)
g = torch.Generator().manual_seed(0)
t0 = time.perf_counter()
for step in range(2000):
    detector.train()
    idx = torch.randint(0, len(X_train_t), (64,), generator=g)
    opt.zero_grad()
    loss = detection_loss(detector(X_train_t[idx].to(device)), obj_t[idx].to(device), cls_t[idx].to(device), box_t[idx].to(device))
    loss.backward()
    opt.step()
print(f"train detector: {time.perf_counter() - t0:.0f}s | loss cuối = {loss.item():.3f}")

### Giải mã và Non-Maximum Suppression

Giải mã: điểm của ô = objectness × xác suất lớp cao nhất; giữ ô có điểm > 0.3, đổi $(o_x, o_y, w, h)$ về toạ độ pixel. Vài ô cạnh nhau có thể cùng "thấy" một vật → **hộp trùng lặp**. **NMS**: sắp xếp hộp theo điểm giảm dần; lấy hộp tốt nhất, loại mọi hộp còn lại **cùng lớp** có IoU với nó > ngưỡng; lặp lại.

In [ ]:
def nms(boxes, scores, iou_threshold=0.5):
    order = np.argsort(-scores)
    keep = []
    while len(order):
        best, order = order[0], order[1:]
        keep.append(best)
        if len(order):
            order = order[box_iou(boxes[best:best + 1], boxes[order])[0] < iou_threshold]
    return np.array(keep, dtype=int)


@torch.no_grad()
def detect(model, X, score_threshold=0.3, use_nms=True):
    """Trả về list mảng (k, 6): x1, y1, x2, y2, score, class."""
    model.eval()
    out = torch.cat([model(X[i:i + 250].to(device)).cpu() for i in range(0, len(X), 250)])
    score, cls = (torch.sigmoid(out[..., :1]) * torch.softmax(out[..., 1:1 + NC], -1)).max(-1)
    box = torch.sigmoid(out[..., 1 + NC:])
    results = []
    for i in range(len(X)):
        gy, gx = torch.nonzero(score[i] > score_threshold, as_tuple=True)
        ox, oy, w, h = box[i, gy, gx].T
        cx, cy = (gx + ox) * CELL, (gy + oy) * CELL
        dets = torch.stack([cx - w * IMG / 2, cy - h * IMG / 2, cx + w * IMG / 2, cy + h * IMG / 2,
                            score[i, gy, gx], cls[i, gy, gx].float()], 1).numpy().reshape(-1, 6)
        if use_nms:
            keep = [j for c in range(NC) for j in np.where(dets[:, 5] == c)[0][nms(dets[dets[:, 5] == c, :4], dets[dets[:, 5] == c, 4])]]
            dets = dets[sorted(keep)]
        results.append(dets)
    return results


dets_nms = detect(detector, X_test_t)
dets_raw = detect(detector, X_test_t, use_nms=False)
n_gt = sum(len(t["labels"]) for t in T_test)
print(f"{n_gt} vật thể thật | {sum(map(len, dets_raw))} hộp trước NMS | {sum(map(len, dets_nms))} hộp sau NMS")
try:
    from torchvision.ops import nms as tv_nms
    d = dets_raw[int(np.argmax([len(x) for x in dets_raw]))]
    same = sorted(nms(d[:, :4], d[:, 4], 0.3)) == sorted(tv_nms(torch.tensor(d[:, :4]), torch.tensor(d[:, 4]), 0.3).tolist())
    print("NMS khớp torchvision.ops.nms:", same)
except ImportError:
    pass

fig, axes = plt.subplots(1, 5, figsize=(14, 3))
for ax, i in zip(axes, range(5)):
    ax.imshow(X_test[i], cmap="gray")
    draw_boxes(ax, T_test[i]["boxes"], T_test[i]["labels"], "lime")
    draw_boxes(ax, dets_nms[i][:, :4], dets_nms[i][:, 5], "red", scores=dets_nms[i][:, 4])
    ax.axis("off")
axes[0].set_title("xanh: thật, đỏ: dự đoán", fontsize=9)
plt.tight_layout()
plt.show()

## 4. Đánh giá: precision–recall, AP, mAP

Với mỗi lớp: gom mọi dự đoán của lớp đó trên **toàn bộ** tập test, sắp theo điểm giảm dần; một dự đoán là **TP** nếu IoU ≥ ngưỡng với một hộp thật cùng lớp **chưa được ghép**, ngược lại là **FP** (kể cả hộp trùng lặp). Đi dọc danh sách → đường precision–recall. **AP** = diện tích dưới đường PR (sau khi làm "precision không tăng" từ phải sang trái); **mAP** = trung bình AP các lớp.

In [ ]:
def average_precision(detections, targets, c, iou_threshold=0.5, return_curve=False):
    records = [(row[4], i, row[:4]) for i, d in enumerate(detections) for row in d[d[:, 5] == c]]
    records.sort(key=lambda r: -r[0])
    n_gt = sum(int((t["labels"] == c).sum()) for t in targets)
    matched, tp = set(), np.zeros(len(records))
    for k, (_, i, b) in enumerate(records):
        gt = targets[i]["boxes"][targets[i]["labels"] == c]
        if len(gt):
            ious = box_iou(b[None], gt)[0]
            j = int(ious.argmax())
            if ious[j] >= iou_threshold and (i, j) not in matched:
                tp[k] = 1
                matched.add((i, j))
    recall = np.cumsum(tp) / n_gt
    precision = np.cumsum(tp) / np.arange(1, len(tp) + 1)
    if return_curve:
        return recall, precision
    r = np.r_[0, recall, 1]
    p = np.r_[1, precision, 0]
    p = np.maximum.accumulate(p[::-1])[::-1]                            # precision không tăng theo recall
    step = np.where(r[1:] != r[:-1])[0]
    return np.sum((r[step + 1] - r[step]) * p[step + 1])


def precision_recall_at_threshold(detections, targets, iou_threshold=0.5):
    """Precision/recall của TẤT CẢ hộp đã giữ (ngưỡng điểm cố định) — cách hệ thống thật dùng kết quả."""
    tp = 0
    for d, t in zip(detections, targets):
        matched = set()
        for row in d[np.argsort(-d[:, 4])]:
            gt = np.where(t["labels"] == row[5])[0]
            if len(gt):
                ious = box_iou(row[None, :4], t["boxes"][gt])[0]
                j = gt[int(ious.argmax())]
                if ious.max() >= iou_threshold and j not in matched:
                    matched.add(j)
                    tp += 1
    return tp / sum(map(len, detections)), tp / sum(len(t["labels"]) for t in targets)


for name, dets in [("có NMS", dets_nms), ("không NMS", dets_raw)]:
    maps = [np.mean([average_precision(dets, T_test, c, thr) for c in range(NC)]) for thr in [0.5, 0.75]]
    p, r = precision_recall_at_threshold(dets, T_test)
    print(f"{name:9s} | mAP@0.5 = {maps[0]:.3f} | mAP@0.75 = {maps[1]:.3f} | tại ngưỡng điểm 0.3: precision = {p:.3f}, recall = {r:.3f}")

fig, ax = plt.subplots(figsize=(5.5, 4))
for c in range(NC):
    r, p = average_precision(dets_nms, T_test, c, return_curve=True)
    ax.plot(r, p, label=SHAPE_CLASSES[c])
ax.set_xlabel("recall")
ax.set_ylabel("precision")
ax.set_title("đường precision–recall theo lớp (IoU ≥ 0.5)")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

- Dữ liệu tổng hợp này **dễ** (vật không chồng nhau, nền đơn giản) nên detector nhỏ đạt mAP gần tuyệt đối — đủ để minh hoạ toàn bộ quy trình. Trên COCO (80 lớp, ảnh thật, vật nhỏ, che khuất), mô hình tốt nhất hiện nay đạt mAP@[0.5:0.95] khoảng 0.6.
- Bỏ NMS gần như **không đổi mAP** ở đây: các hộp trùng có điểm thấp hơn hầu hết hộp đúng nên nằm cuối danh sách xếp hạng — AP chủ yếu phạt các FP xếp **trước** TP. Nhưng hệ thống thật dùng **một ngưỡng điểm cố định**: ở ngưỡng 0.3, không NMS cho thêm vài chục hộp thừa → precision giảm, và mọi phép **đếm** vật thể bị sai. Với detector thật (hàng nghìn anchor mỗi ảnh, nhiều hộp chồng lên một vật), NMS là bắt buộc (trừ DETR — dự đoán tập hợp, ghép cặp một-một khi train). Bài học phụ: một chỉ số tổng hợp (mAP) có thể che khuất vấn đề mà người dùng cuối thấy rõ.

### Các họ detector thực tế

| Họ | Ý tưởng | Ví dụ |
|---|---|---|
| Hai giai đoạn | đề xuất vùng (RPN) → phân loại + tinh chỉnh từng vùng | Faster R-CNN, Mask R-CNN |
| Một giai đoạn, anchor | lưới + nhiều hộp mẫu (anchor) mỗi ô; focal loss cho mất cân bằng nền/vật | SSD, RetinaNet, YOLOv3–v5 |
| Một giai đoạn, không anchor | dự đoán tâm/khoảng cách tới cạnh tại mỗi vị trí | CenterNet, FCOS, YOLOv8+ |
| Transformer | truy vấn học được + ghép cặp Hungarian, không NMS | DETR, RT-DETR |

Detector của bài là phiên bản tối giản của YOLO gốc (1 hộp/ô, không anchor). Bản thật thêm: nhiều tỉ lệ (FPN) cho vật lớn/nhỏ, nhiều hộp mỗi ô, loss hộp theo IoU (GIoU/CIoU), augmentation mạnh (mosaic).

## 5. Phân đoạn ngữ nghĩa với U-Net

Phân loại **từng pixel** (6 lớp: nền + 5 hình). **U-Net** (Ronneberger et al., 2015): nhánh **mã hoá** giảm kích thước để thu ngữ cảnh, nhánh **giải mã** phóng to lại; **skip connection** nối đặc trưng độ phân giải cao của nhánh mã hoá sang nhánh giải mã cùng mức — giữ chi tiết biên. So sánh có và không có skip:

In [ ]:
class UNet(nn.Module):
    def __init__(self, skip=True, n_classes=NC + 1):
        super().__init__()
        self.skip = skip

        def block(i, o):
            return nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(),
                                 nn.Conv2d(o, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU())

        self.enc1, self.enc2, self.enc3 = block(1, 16), block(16, 32), block(32, 64)
        self.pool = nn.MaxPool2d(2)
        self.up2, self.up1 = nn.ConvTranspose2d(64, 32, 2, stride=2), nn.ConvTranspose2d(32, 16, 2, stride=2)
        self.dec2 = block(64 if skip else 32, 32)
        self.dec1 = block(32 if skip else 16, 16)
        self.out = nn.Conv2d(16, n_classes, 1)

    def forward(self, x):
        e1 = self.enc1(x)                                               # 64×64
        e2 = self.enc2(self.pool(e1))                                   # 32×32
        e3 = self.enc3(self.pool(e2))                                   # 16×16
        d2 = self.up2(e3)
        d2 = self.dec2(torch.cat([d2, e2], 1) if self.skip else d2)
        d1 = self.up1(d2)
        d1 = self.dec1(torch.cat([d1, e1], 1) if self.skip else d1)
        return self.out(d1)


def segmentation_metrics(pred, true, n_classes=NC + 1):
    ious = [((pred == c) & (true == c)).sum().item() / ((pred == c) | (true == c)).sum().item()
            for c in range(n_classes) if ((pred == c) | (true == c)).any()]
    return {"pixel accuracy": (pred == true).float().mean().item(), "mIoU": float(np.mean(ious)),
            "pixel sai / ảnh": (pred != true).sum().item() / len(true)}


M_train_t, M_test_t = torch.tensor(M_train), torch.tensor(M_test)
seg_preds = {}
base = segmentation_metrics(torch.zeros_like(M_test_t), M_test_t)
print(f"chỉ đoán 'nền'    : pixel accuracy = {base['pixel accuracy']:.4f} | mIoU = {base['mIoU']:.4f} | pixel sai / ảnh = {base['pixel sai / ảnh']:.1f}")
for skip in [False, True]:
    torch.manual_seed(0)
    net = UNet(skip).to(device)
    opt = torch.optim.Adam(net.parameters(), lr=1e-3)
    g = torch.Generator().manual_seed(0)
    t0 = time.perf_counter()
    for step in range(1500):
        net.train()
        idx = torch.randint(0, len(X_train_t), (32,), generator=g)
        opt.zero_grad()
        F.cross_entropy(net(X_train_t[idx].to(device)), M_train_t[idx].to(device)).backward()
        opt.step()
    net.eval()
    with torch.no_grad():
        seg_preds[skip] = torch.cat([net(X_test_t[i:i + 100].to(device)).argmax(1).cpu() for i in range(0, len(X_test_t), 100)])
    m = segmentation_metrics(seg_preds[skip], M_test_t)
    print(f"U-Net {'có' if skip else 'không':5s} skip: " + " | ".join(f"{k} = {v:.4f}" if k != "pixel sai / ảnh" else f"{k} = {v:.1f}"
                                                                for k, v in m.items()) + f" | {time.perf_counter() - t0:.0f}s")

In [ ]:
i = int(np.argmax([(seg_preds[False][k] != M_test_t[k]).sum().item() for k in range(50)]))
fig, axes = plt.subplots(1, 5, figsize=(14, 3))
axes[0].imshow(X_test[i], cmap="gray")
axes[0].set_title("ảnh")
axes[1].imshow(M_test[i], cmap="tab10", vmin=0, vmax=9, interpolation="nearest")
axes[1].set_title("mask thật")
for ax, skip in zip(axes[2:4], [False, True]):
    ax.imshow(seg_preds[skip][i], cmap="tab10", vmin=0, vmax=9, interpolation="nearest")
    ax.set_title(f"dự đoán ({'có' if skip else 'không'} skip)")
axes[4].imshow((seg_preds[False][i] != M_test_t[i]).float() - (seg_preds[True][i] != M_test_t[i]).float(), cmap="bwr", vmin=-1, vmax=1)
axes[4].set_title("đỏ: pixel chỉ sai khi không skip")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

- **Pixel accuracy đánh lừa**: đoán toàn bộ là "nền" đã đạt khoảng 93% pixel accuracy (nền chiếm đa số) nhưng mIoU rất thấp. Dùng **mIoU** (trung bình IoU theo lớp) hoặc Dice; với vật nhỏ, xem thêm các chỉ số theo biên.
- Cả hai mạng đều tốt trên dữ liệu đơn giản này, nhưng bản **không skip** sai nhiều pixel hơn hẳn — và lỗi nằm ở **biên** vật thể: thông tin vị trí chi tiết mất khi giảm xuống 16×16 không thể khôi phục chỉ bằng phóng to. Skip connection mang chi tiết đó trực tiếp sang nhánh giải mã.

### Phân đoạn thực thể và mô hình nền tảng

- **Instance segmentation**: Mask R-CNN = Faster R-CNN + một nhánh dự đoán mask cho mỗi hộp.
- **Segment Anything (SAM)**: mô hình nền tảng, nhận "gợi ý" (điểm, hộp) và trả về mask của bất kỳ vật thể nào — kể cả loại chưa từng thấy; thường dùng để **gán nhãn nhanh** dữ liệu phân đoạn.
- Thư viện: torchvision (`fasterrcnn_resnet50_fpn`, `maskrcnn_resnet50_fpn`, `deeplabv3`), Ultralytics YOLO, Detectron2, MMDetection; gán nhãn: CVAT, Label Studio.

In [ ]:
RUN_PRETRAINED = False                                                 # cần mạng để tải trọng số (~160 MB)

if RUN_PRETRAINED:
    from torchvision.models.detection import FasterRCNN_ResNet50_FPN_V2_Weights, fasterrcnn_resnet50_fpn_v2

    weights = FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT
    model = fasterrcnn_resnet50_fpn_v2(weights=weights).eval()
    image = torch.rand(3, 480, 640)                                    # thay bằng ảnh thật (C, H, W) trong [0, 1]
    with torch.no_grad():
        out = model([weights.transforms()(image)])[0]                  # dict: boxes (xyxy), labels, scores
    print({k: tuple(v.shape) for k, v in out.items()}, [weights.meta["categories"][i] for i in out["labels"][:5]])
else:
    print("Bỏ qua phần mô hình tiền huấn luyện thật (RUN_PRETRAINED = False).")

## 🧠 Kiểm tra nhanh

1. Phát hiện vật thể khác định vị và phân đoạn thế nào?
2. Tính IoU của hai hộp. Vì sao IoU khắt khe với vật nhỏ?
3. NMS làm gì? Điều gì xảy ra với AP nếu bỏ NMS?
4. Mô tả cách tính AP cho một lớp. mAP@0.5 khác mAP@[0.5:0.95] thế nào?
5. Detector dạng lưới gán "trách nhiệm" cho ô nào? Hạn chế khi hai vật có tâm trong cùng một ô?
6. Vì sao pixel accuracy không phù hợp để đánh giá phân đoạn? Skip connection của U-Net giúp gì?

## 📌 Tóm tắt

- Detection = hộp + nhãn + điểm cho số lượng vật tuỳ ý; đo khớp bằng IoU; khử trùng lặp bằng NMS; đánh giá bằng AP/mAP trên đường precision–recall.
- Detector lưới (YOLO): mỗi ô dự đoán objectness, lớp, hộp; các detector thật thêm anchor/đa tỉ lệ/loss IoU hoặc dùng Transformer (DETR).
- Phân đoạn ngữ nghĩa = phân loại từng pixel; U-Net với skip connection giữ chi tiết biên; đánh giá bằng mIoU/Dice thay vì pixel accuracy.

## 📚 Tài liệu tham khảo

- Redmon et al. (2016) — *You Only Look Once: Unified, Real-Time Object Detection*
- Ren et al. (2015) — *Faster R-CNN*; He et al. (2017) — *Mask R-CNN*
- Lin et al. (2017) — *Focal Loss for Dense Object Detection* (RetinaNet)
- Carion et al. (2020) — *End-to-End Object Detection with Transformers* (DETR)
- Ronneberger, Fischer & Brox (2015) — *U-Net*
- Kirillov et al. (2023) — *Segment Anything*
- Padilla et al. (2020) — *A Survey on Performance Metrics for Object-Detection Algorithms*